# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tushar-sharma001/Flyrank-Ml-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Scoring / Ranking.**

The decision from ML-02 was "which pages should a reviewer look at first" — a
"which ones first?" question, which the framing-ml-problems skill maps directly
to Ranking/Scoring, with a priority score as the target and precision@K as the
metric. Under the hood this is built on a classification-style signal (probability
of decline), then combined with baseline signals into one ranked score — this
mirrors the lane guide's own approach in Section 5, where a model probability and
a normalized baseline score are blended into `final_refresh_score`. But the thing
I'm actually optimizing and delivering is a ranked list, not a yes/no label, so
Scoring/Ranking is the right primary type to name.

In [8]:
import os
if not os.path.exists("Flyrank-Ml-Internship"):
    !git clone https://github.com/tushar-sharma001/Flyrank-Ml-Internship.git

import pandas as pd
from pathlib import Path

candidates = [Path("data/raw/content_refresh_anonymized.csv"),
              Path("../../data/raw/content_refresh_anonymized.csv"),
              Path("Flyrank-Ml-Internship/data/raw/content_refresh_anonymized.csv")]
csv_path = next(p for p in candidates if p.exists())

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy (not a true future outcome):** `is_declining_label = (trend_direction == "down")`.

This is the only label available in the starter snapshot, and it's honestly a
current-window proxy, not an observed future outcome — trend_direction describes
what already happened in this window, not what happens next. The lane guide flags
this exact weakness in Section 5: it's a fine label to prove the workflow end to
end, but a stronger capstone version would define the target as a genuine
future-window outcome (e.g. decline over the *next* 30 days, using the warehouse's
daily facts) once I move past the starter data.

In [9]:
df = pd.read_csv(csv_path)
df_f = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").copy()
df_f["is_declining_label"] = (df_f["trend_direction"] == "down").astype(int)

print(f"Rows: {len(df_f)}")
print(df_f['is_declining_label'].value_counts())
print(f"Positive rate: {df_f['is_declining_label'].mean():.3f}")

Rows: 30000
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Positive rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50** — matching the reviewer capacity assumed in ML-02 (a few dozen
pages reviewed per week). "Good" means: of the top 50 pages my ranking surfaces,
a high share are genuinely in the group that needs attention.

The naive floor to beat: since 54.2% of pages in this filtered slice already show
`trend_direction == "down"`, picking 50 pages at random would already look
deceptively decent on this simple proxy — so precision@50 has to clear a
meaningfully higher bar than blind guessing to mean anything.

For a stronger reference point, the lane guide's own documented pipeline
(Section 5, on its own refined score and label) shows a rule-based baseline
scoring precision@50 of 0.240, versus 0.740 for a trained random forest — real
evidence, from FlyRank's own numbers, that a learned ranking can beat a fixed
rule by a wide margin, even though that specific label differs from my simple
proxy here.

In [10]:
naive_floor = df_f["is_declining_label"].mean()
print(f"Naive precision@50 floor (base rate on this proxy): {naive_floor:.3f}")
print("Any real ranking needs to clear this by a meaningful margin to prove value.")

Naive precision@50 floor (base rate on this proxy): 0.542
Any real ranking needs to clear this by a meaningful margin to prove value.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one piece of content, for one client, in this 90-day window** —
already deduplicated by `content_id` per the lane guide's standard filter. Grain
confirmed below: no `content_id` appears more than once.

In [11]:
grain_check = df_f.groupby("content_id").size()
print("Max rows per content_id (should be 1):", grain_check.max())

df_f[["content_id", "client_id", "trend_direction", "is_declining_label",
      "impressions_90d", "avg_position", "ctr", "days_since_last_update"]].head(10)

Max rows per content_id (should be 1): 1


,content_id,client_id,trend_direction,is_declining_label,impressions_90d,avg_position,ctr,days_since_last_update
0,content_304f48230142,client_f369cb89fc,down,1,3803,10.6,0.76,20
1,content_a1fb4e703a9e,client_4e07408562,down,1,15320,20.3,0.05,25
2,content_9aa793d4d895,client_7f2253d7e2,down,1,12581,36.5,0.09,20
3,content_331d6c4de07b,client_19581e27de,stable,0,11751,6.2,0.49,22
4,content_d99b7a2d90ca,client_3fdba35f04,down,1,19140,44.0,0.13,14
5,content_d4084a4bc775,client_f369cb89fc,down,1,3970,8.5,0.03,20
6,content_9a34b442b552,client_8722616204,down,1,20,7.0,0.00,20
7,content_a63219c6e95a,client_19581e27de,stable,0,1724,21.2,0.06,22
8,content_5e6c160719bc,client_6208ef0f77,down,1,32574,46.0,0.09,20
9,content_c27558df2b0c,client_19581e27de,down,1,1240,4.9,0.16,104


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

No single observable signal separates decliners from non-decliners cleanly — the
strongest individual correlation with the label among safe candidate features is
only about 0.16 (content_age_days), and everything else is weaker still. That
means a simple if-statement threshold on any one column would perform poorly;
the real pattern, if it exists, is spread thinly across many weakly-informative
signals at once — exactly the "many signals, tangled" case the framing-ml-problems
skill says is where ML earns its place over a plain rule. The lane guide's own
0.240-vs-0.740 precision@50 gap (Section 5) is further evidence in the same
direction: FlyRank's own rule-based baseline underperforms a model that combines
signals, on their own data.

(Note: `trend_direction`/`trend_pct` are deliberately excluded from this
correlation check — they're the source of the label itself, and using them as a
"feature" would be the exact label-trap leakage the flyrank-data skill warns
against.)

In [12]:
feature_cols = ["avg_position", "ctr", "impressions_90d", "days_since_last_update",
                "word_count", "engagement_rate", "scroll_rate", "search_volume",
                "competition", "content_age_days"]

for col in feature_cols:
    corr = df_f[col].corr(df_f["is_declining_label"])
    print(f"{col}: corr with label = {corr:.3f}")

print()
print("Max abs correlation among candidate features:",
      round(max(abs(df_f[c].corr(df_f['is_declining_label'])) for c in feature_cols), 3))

avg_position: corr with label = -0.029
ctr: corr with label = -0.062
impressions_90d: corr with label = -0.018
days_since_last_update: corr with label = 0.081
word_count: corr with label = 0.090
engagement_rate: corr with label = -0.013
scroll_rate: corr with label = -0.003
search_volume: corr with label = -0.019
competition: corr with label = -0.009
content_age_days: corr with label = -0.164

Max abs correlation among candidate features: 0.164


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.